# 01. Single-Agent Foundations

A first runnable notebook for the OpenAI Agents SDK. It uses a tiny digital humanities example: extracting people, places, dates, and evidence from short historical texts.

## Run locally
1. `uv sync`
2. set `OPENAI_API_KEY`
3. run the cells top to bottom

## Google Colab
1. Open a new Colab notebook
2. Run `!pip install openai-agents pandas`
3. Paste the cells from this notebook
4. Add your API key with `os.environ["OPENAI_API_KEY"] = "..."` or use Colab secrets

## Concepts
- Agent
- Tools
- Structured output
- Reproducibility

## Dataset
A few short DH-flavored sample texts are included in `notebooks/common.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOK_DIR))

from common import LETTER_TEXTS
from agents import Agent, Runner, function_tool

print('loaded', len(LETTER_TEXTS), 'sample texts')

## Step 1: A small tool

Tools are ordinary Python functions. Here the tool normalizes a place name so the agent can reuse it consistently.

In [ ]:
@function_tool
def normalize_place_name(place: str) -> str:
    """Normalize a place name for historical records."""
    return place.strip().title()

print(normalize_place_name(' madrid '))

## Step 2: Define the output shape

Using a structured output makes it easy to inspect results and grade student work.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Extraction:
    people: list[str] = field(default_factory=list)
    places: list[str] = field(default_factory=list)
    dates: list[str] = field(default_factory=list)
    evidence: list[str] = field(default_factory=list)


## Step 3: Build the agent

The agent is intentionally narrow: it extracts evidence from short texts. The default model will be used unless you override it.

In [ ]:
archive_agent = Agent(
    name='Archive Extractor',
    instructions=(
        'Extract people, places, dates, and evidence from short historical text. '
        'Be conservative: if you are unsure, leave a field empty rather than guessing.'
    ),
    tools=[normalize_place_name],
    output_type=Extraction,
)

archive_agent

## Step 4: Run the agent

Set `text` to one of the sample rows. If you do not have an API key yet, this cell is the one to revisit after setup.

In [ ]:
text = LETTER_TEXTS[0]['text']
result = Runner.run_sync(archive_agent, text)
print(result.final_output)


## Step 5: Put results in a table

This is useful for classroom discussion and for comparing outputs across prompt versions.

In [ ]:
rows = [
    {'field': 'people', 'value': result.final_output.people},
    {'field': 'places', 'value': result.final_output.places},
    {'field': 'dates', 'value': result.final_output.dates},
    {'field': 'evidence', 'value': result.final_output.evidence},
]
pd.DataFrame(rows)

## Exercise

Change the instructions so the agent extracts institutions instead of people. Use the printing house notice and compare results.

## Solution

Replace `people` with `institutions` in the output schema, update the instructions to say `extract institutions`, and rerun on `LETTER_TEXTS[1]`.